In [1]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

C:\Users\user\AppData\Local\Temp\ipykernel_7212\3594771651.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
docs = (
    PyPDFLoader("../documents/Company_Policies.pdf").load() + 
    PyPDFLoader("../documents/Company_Profile.pdf").load() + 
    PyPDFLoader("../documents/Product_and_Pricing.pdf").load() 
)

In [3]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, 
    chunk_overlap=150, 
).split_documents(docs)

In [ ]:
embeddings = OllamaEmbeddings(model='bge-m3')
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../vector-database/selfrag3"
)

In [ ]:
llm = ChatOllama(model="llama3.2", temperature=0)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})



In [ ]:
# Graph state 
class State(TypedDict): 
    question: str
    need_retrieval: bool

    docs: List[Document]
    relevant_docs: List[Document]

    context: str
    answer: str